# 03 · Dataset final `data/processed/usab_v2`

Dos niveles enlazados por rasgos DOM. Ver `docs/metodologia.md` y `data_dictionary.csv`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[0] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.max_columns', 80); pd.set_option('display.width', 200)
from usab_ds import config

In [2]:
P = config.PROCESSED
sessions = pd.read_parquet(P / 'sessions.parquet')
labels = pd.read_parquet(P / 'heuristic_evaluations.parquet')
iface = pd.read_parquet(P / 'interface_labels.parquet')
pages = pd.read_parquet(P / 'pages_static.parquet')
dd = pd.read_csv(P / 'data_dictionary.csv')
{k: v.shape for k, v in dict(sessions=sessions, labels=labels, iface=iface, pages=pages).items()}

{'sessions': (22836, 91),
 'labels': (1715, 17),
 'iface': (95, 49),
 'pages': (890, 28)}

In [3]:
main = sessions[sessions.include_main]   # humanos con quality_ok
main.groupby(['source','user_group','assistive_tech_participant'], dropna=False).size()

source                     user_group  assistive_tech_participant
S01_web_emotions_28489601  unknown     NaN                           11210
S02_a11y_cua               BLV         screen_reader                   105
                                       unknown                          81
                           sighted     none                            167
S08_dashboards             unknown     NaN                              50
dtype: int64

In [4]:
iso = [c for c in sessions.columns if c.startswith('iso_')]
main.groupby('source')[iso].describe().T

source                                   S01_web_emotions_28489601 S02_a11y_cua S08_dashboards
iso_efficiency_task_time_s         count                   11210.0        353.0           50.0
                                   mean                 187.158133   184.010595     6039.92688
                                   std                  559.507169   160.093032    9378.659358
                                   min                       5.001       21.573       1031.124
                                   25%                     12.6325       66.362      3034.2325
                                   50%                     33.6795      132.832       3920.289
                                   75%                    115.6555      247.228     6114.20175
                                   max                   22926.746       827.28      67026.618
iso_efficiency_actions             count                   11210.0        353.0           50.0
                                   mean                  34.901606   108.271955        2450.26
                                   std                  108.447995   118.284541    2163.711039
                                   min                         0.0          1.0          717.0
                                   25%                         2.0         34.0        1313.25
                                   50%                         7.0         64.0         1799.0
                                   75%                        23.0        139.0        2781.75
                                   max                      2487.0        842.0        12381.0
iso_effectiveness_task_completion  count                       0.0        353.0            0.0
                                   mean                       <NA>      0.92068           <NA>
                                   std                        <NA>     0.270621           <NA>
                                   min                        <NA>          0.0           <NA>
                                   25%                        <NA>          1.0           <NA>
                                   50%                        <NA>          1.0           <NA>
                                   75%                        <NA>          1.0           <NA>
                                   max                        <NA>          1.0           <NA>
iso_efficiency_time_based          count                       0.0        353.0            0.0
                                   mean                       <NA>      0.62769           <NA>
                                   std                        <NA>     0.532003           <NA>
                                   min                        <NA>          0.0           <NA>
                                   25%                        <NA>     0.242691           <NA>
                                   50%                        <NA>     0.451698           <NA>
                                   75%                        <NA>     0.904132           <NA>
                                   max                        <NA>     2.781254           <NA>
iso_satisfaction_perceived_ease    count                       0.0        353.0            0.0
                                   mean                       <NA>     0.653683           <NA>
                                   std                        <NA>     0.169392           <NA>
                                   min                        <NA>         0.25           <NA>
                                   25%                        <NA>          0.5           <NA>
                                   50%                        <NA>         0.75           <NA>
                                   75%                        <NA>         0.75           <NA>
                                   max                        <NA>         0.75           <NA>
iso_satisfaction_affective_valence count                    5354.0          0.0            0.0
          

In [5]:
proxy = [f'{h}_proxy_z' for h in config.TARGET_HEURISTICS]
main.groupby(['source','user_group'])[proxy].median()

H1_proxy_z  H4_proxy_z    H5_proxy_z  H6_proxy_z
source                    user_group                                                  
S01_web_emotions_28489601 unknown       0.000000         NaN           NaN    0.094876
S02_a11y_cua              BLV                NaN   -0.285462           NaN    0.616995
                          sighted       0.063983    0.586280           NaN    0.327358
S08_dashboards            unknown       0.131852    0.000000  1.318390e-16    0.092410

In [6]:
iface.filter(regex='^(source|interface_name|evaluator_type|H[1456]_violation$|H[1456]_violation_rate|dom_pages)').head(30)

,source,interface_name,evaluator_type,H1_violation,H1_violation_rate,H4_violation,H4_violation_rate,H5_violation,H5_violation_rate,H6_violation,H6_violation_rate,dom_pages
0,S03_vt_llm_prototypes_29042675,Bakery | Baking shop,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
1,S03_vt_llm_prototypes_29042675,E tutor | E-Tutor,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
2,S03_vt_llm_prototypes_29042675,Food delivery,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
3,S03_vt_llm_prototypes_29042675,Food Order,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
4,S03_vt_llm_prototypes_29042675,GAIA community app | Gaia,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
5,S03_vt_llm_prototypes_29042675,Hospital App 1 | HospitalApp,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
6,S03_vt_llm_prototypes_29042675,Hospital App 2 | HospitalApp2,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
7,S03_vt_llm_prototypes_29042675,Housing,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
8,S03_vt_llm_prototypes_29042675,Notification,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN
9,S03_vt_llm_prototypes_29042675,Running | Running app,llm,1.0,1.0,1.0,1.000000,1.0,1.000000,1.0,1.000000,NaN


In [7]:
pd.read_csv(P / 'reliability.csv').query('heuristic in ["H1","H4","H5","H6","all"]')

,source,heuristic,evaluator_type,n_interfaces,prevalence,krippendorff_alpha
0,S03_vt_llm_prototypes_29042675,H1,llm,13,1.000,NaN
4,S03_vt_llm_prototypes_29042675,H4,llm,14,1.000,NaN
5,S03_vt_llm_prototypes_29042675,H5,llm,14,1.000,NaN
6,S03_vt_llm_prototypes_29042675,H6,llm,18,1.000,NaN
10,S04_catching_ux_flaws,H1,llm,30,0.966,0.658824
14,S04_catching_ux_flaws,H4,llm,30,0.795,0.102381
15,S04_catching_ux_flaws,H5,llm,30,0.876,0.384615
16,S04_catching_ux_flaws,H6,llm,30,0.618,0.152941
20,S06_p3m,H1,llm,0,0.400,NaN
21,S06_p3m,H1,llm_expert_validated,0,0.300,NaN


In [8]:
dd[dd.table == 'sessions'][['column','description','iso_characteristic','heuristic','evidence','non_null_pct']]

,column,description,iso_characteristic,heuristic,evidence,non_null_pct
0,source,Identificador de la fuente (configs/sources.ya...,NaN,NaN,NaN,100.0
1,session_id,Identificador único de sesión <fuente>:<id loc...,NaN,NaN,NaN,100.0
2,participant_id,Participante (seudónimo de la fuente) o nombre...,NaN,NaN,NaN,100.0
3,user_group,BLV (ciego/baja visión) | sighted | agent | un...,NaN,NaN,NaN,100.0
4,is_human,False para agentes de uso de computadora (CUA).,NaN,NaN,NaN,100.0
...,...,...,...,...,...,...
86,H5_proxy_z,Índice exploratorio: media de z de los indicad...,NaN,H5,operational,0.2
87,H5_proxy_n_indicators,Indicadores disponibles para el índice proxy.,NaN,H5,NaN,100.0
88,H6_proxy_z,Índice exploratorio: media de z de los indicad...,NaN,H6,operational,78.9
89,H6_proxy_n_indicators,Indicadores disponibles para el índice proxy.,NaN,H6,NaN,100.0
